# Cash File Caching and Variable Tracking Demo

This notebook demonstrates two key features of the `cash` library:
1. **Automatic File Tracking:** `cash` detects when you read files (via `open` or `pandas`) and automatically invalidates the cache if those files change.
2. **Variable Caching & Lineage:** It tracks variable dependencies across cells, avoiding re-execution if inputs haven't changed.

In [ ]:
import pandas as pd

In [ ]:
%cash_on

## 1. File Tracking
Run the cell below. It reads `sales_data.csv`. The first time, it will execute. If you run it again immediately, it should use the cache.

In [ ]:
print("Reading sales data...")
# This read operation is automatically tracked by cash
df = pd.read_csv('sales_data.csv')
print(f"Loaded {len(df)} rows.")
df.head()

### Test Invalidation
1. Open `sales_data.csv` in a text editor.
2. Add a new row (e.g., `2023-01-07,East,WidgetC,300,10`) and save.
3. Re-run the cell above.
4. **Result:** You should see "Reading sales data..." print again, indicating a cache miss because the file changed.

## 2. Variable Tracking & Lineage
The following cells depend on `df`. If `df` is restored from cache (and its lineage hash is unchanged), these downstream cells should also use their cache.

In [ ]:
print("Calculating regional summary...")
# Depends on 'df'
regional_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
regional_sales

In [ ]:
print("Calculating product summary...")
# Depends on 'df'
product_sales = df.groupby('Product')['Units'].mean()
product_sales

### Test Downstream Caching
1. Do NOT modify the CSV.
2. Re-run the `Reading sales data...` cell (it should hit cache).
3. Re-run the summary cells above.
4. **Result:** They should hit the cache (no print output, fast execution) because `df` is logically the same.